# 02 — Data Cleaning & Validation
    Create reproducible cleaning steps, validate domains and preserve raw files unchanged.

In [ ]:
from pathlib import Path
    import pandas as pd
    RAW = Path("../01_Raw_Data")
    OUT = Path("../04_Data_Cleaning/cleaned_data")
    OUT.mkdir(exist_ok=True)
    employees = pd.read_csv(RAW/"employees.csv", parse_dates=["hire_date"])
    training = pd.read_csv(RAW/"training_records.csv", parse_dates=["enrollment_date"])
    performance = pd.read_csv(RAW/"performance_reviews.csv")
    attendance = pd.read_csv(RAW/"monthly_attendance.csv")
    recruitment = pd.read_csv(RAW/"recruitment_pipeline.csv", parse_dates=["application_date"])

In [ ]:
frames = {"employees":employees, "training":training, "performance":performance,
              "attendance":attendance, "recruitment":recruitment}
    for name, df in frames.items():
        df.drop_duplicates(inplace=True)
        for col in df.select_dtypes(include="object").columns:
            df[col] = df[col].str.strip()
    employees["department"] = employees["department"].str.title()
    employees["employment_status"] = employees["employment_status"].str.title()
    employees["age"] = pd.to_numeric(employees["age"], errors="coerce")
    employees["annual_salary_inr"] = pd.to_numeric(employees["annual_salary_inr"], errors="coerce")
    employees["satisfaction_score"] = employees["satisfaction_score"].clip(1,5)
    employees["engagement_score"] = employees["engagement_score"].clip(0,100)
    performance["performance_rating"] = performance["performance_rating"].clip(1,5)
    training["assessment_score"] = training["assessment_score"].clip(0,100)
    attendance["days_absent"] = attendance["days_absent"].clip(lower=0)
    attendance["late_arrivals"] = attendance["late_arrivals"].clip(lower=0)

In [ ]:
cleaned = {"employees":employees, "training_records":training,
               "performance_reviews":performance, "monthly_attendance":attendance,
               "recruitment_pipeline":recruitment}
    for name, df in cleaned.items():
        df.to_csv(OUT/f"{name}.csv", index=False)
    quality = pd.DataFrame([{
        "dataset":name, "rows":len(df), "columns":df.shape[1],
        "duplicate_rows":int(df.duplicated().sum()),
        "missing_cells":int(df.isna().sum().sum())
    } for name,df in cleaned.items()])
    display(quality)
    quality.to_csv("../04_Data_Cleaning/data_quality_report.csv", index=False)

## Validation checks
    These assertions are intentionally explicit. Investigate failures rather than silently coercing business data.

In [ ]:
assert employees["employee_id"].is_unique
    assert employees["age"].between(18,100).all()
    assert employees["satisfaction_score"].dropna().between(1,5).all()
    assert employees["engagement_score"].dropna().between(0,100).all()
    assert set(employees["employment_status"].dropna().unique()).issubset({"Active","Exited"})
    assert training["employee_id"].isin(employees["employee_id"]).all()
    print("Core validation checks passed.")